# Lab 15 · FastAPI retrieval service
## Goal
Build and test a typed HTTP endpoint backed by actual pgvector retrieval. Practice request validation, dependency injection, API-key tenant mapping, lifespan resource management, connection pools, and failure handling. Prerequisites: Labs 12–14 and HTTP basics. Plan 120 minutes: 20 service exploration, 45 implementation, 30 tests, 25 server exercise/report.

This is a local teaching service. Example API keys are public, not production credentials. There is no database row-level-security policy in this starter; the application must include the tenant filter on every retrieval path.

## Setup
Complete the track README preflight first: install this track's pinned environment and editable package; start PostgreSQL/pgvector with Docker Compose; download the two pinned model snapshots; seed the invented course corpus. Select that Python environment as the kernel. These notebooks need the **whole `labs/rag` folder**, not a standalone upload. They use a real database and local neural models. Missing services or models are errors, not silently replaced by mocks.

Run worked cells first, then implement TODOs, set `RUN_EXERCISES=True`, restart, and run all. Instructor copies enable those checks. `RUN_GENERATION` stays false unless you deliberately enable the separately documented Gemini request. Offline here means no inference API calls after package/model downloads; PostgreSQL is still a real local service. All data are invented and English-language. Never use this teaching database or its public example keys for real student records.

In [1]:
import json
from pathlib import Path
from importlib.metadata import version
import numpy as np
from raglab.config import ROOT, DIMENSION
from raglab import db, models
from raglab.ingestion import corpus, chunk_documents, make_pipeline
RUN_EXERCISES = False
RUN_GENERATION = False
print({p:version(p) for p in ("llama-index-core","pgvector","psycopg","fastapi","sentence-transformers")})
with db.connect() as conn:
    print(conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone())

{'llama-index-core': '0.14.25', 'pgvector': '0.5.0', 'psycopg': '3.3.6', 'fastapi': '0.142.4', 'sentence-transformers': '6.1.0'}
{'extversion': '0.8.1'}


## Steps
### 1. Start and inspect the reference application
Read `raglab/service.py`. Its lifespan loads models and opens a database pool once. Requests run synchronous inference/database work in FastAPI's synchronous endpoint execution path. A two-slot semaphore rejects excess concurrent work; SQL has a statement timeout. This does not implement an end-to-end CPU inference deadline.

`TestClient` below executes the real ASGI application and database calls in-process. It does not establish that a network server is listening. Use the terminal exercise later for that.

In [2]:
from fastapi.testclient import TestClient
from raglab.service import create_app,SearchRequest,SearchResponse,Hit
app=create_app({"student-alpha":"alpha","student-beta":"beta"})
headers={"X-API-Key":"student-alpha"}
with TestClient(app) as client:
    assert client.get("/health").status_code==200
    response=client.post("/search",headers=headers,json={"query":"camera loans","k":3,"rerank":True})
    assert response.status_code==200,response.text
    print(response.json())
    assert client.post("/search",json={"query":"camera"}).status_code==401
    assert client.post("/search",headers=headers,json={"query":"   "}).status_code==422
    assert client.post("/search",headers=headers,json={"query":"camera","tenant":"beta"}).status_code==422

/private/tmp/course-rag/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
/private/tmp/course-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11760.58it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7333.31it/s]

{'hits': [{'chunk_id': '724b22bcf4eb36e1ce99c1b66ecc413e9e6e946ee3d7d4a69a2db11b328a2bef', 'doc_id': 'camera-extension', 'text': 'Camera loans can be extended by two days only if another student has not reserved the equipment. Request an extension before the original due date. The media desk must approve an extension; submitting a request does not automatically extend a loan.'}, {'chunk_id': 'fc606b6ea46104ec2e48d05565d0233f7c5ed3ff8386b62628fd107b0b63fd6c', 'doc_id': 'camera-loans', 'text': 'Cameras can be borrowed for three days. Bring a valid student card to collect the camera. Overnight loans count toward the three-day period. Return the camera to the media desk before closing.'}, {'chunk_id': '6c741d4438701263f36f6adcbc8cf46289b433ad3b0cd4095152721ad78585ec', 'doc_id': 'battery-guide', 'text': 'The Orion camera battery lasts approximately eight hours under the fictional standard test conditions used in this course. Cold weather and continuous recording can reduce runtime. A spare 

### 2. TODO: add your own FastAPI route (40 points)
Implement `add_student_route(app)` to register POST `/student-search` with request type `SearchRequest` and response type `SearchResponse`. Obtain the tenant through `Depends(app.state.tenant_dependency)`; use the request's application pool and the shared `retrieve` function. Do not accept a tenant in the request body or query parameters. Add the route before starting `TestClient`. This first route intentionally lacks the reference route's load/error policy; add those in your submission after making the data path work.

In [3]:
def add_student_route(app):
    raise NotImplementedError("Register a typed route and inject the authenticated tenant")

In [4]:
if RUN_EXERCISES:
    student_app=create_app({"student-alpha":"alpha","student-beta":"beta"})
    add_student_route(student_app)
    with TestClient(student_app) as client:
        assert client.post("/student-search",json={"query":"camera"}).status_code==401
        response=client.post("/student-search",headers=headers,json={"query":"BETA-ONLY-MARKER camera","k":20})
        assert response.status_code==200,response.text
        assert all("BETA-ONLY-MARKER" not in h["text"] for h in response.json()["hits"])
        assert client.post("/student-search",headers=headers,json={"query":"camera","k":True}).status_code==422
        assert "/student-search" in client.get("/openapi.json").json()["paths"]
    print("Student route exercised against pgvector")

### 3. TODO: write an API contract test (30 points)
Implement `assert_contract(client,path)` using real HTTP test-client calls. Verify missing/invalid keys return 401; a blank query, k=0, and an extra tenant field return 422; a valid alpha query returns 200 with only `hits`, `mode`, `reranked`; no returned text contains the beta marker. The function should raise assertions on a regression and return nothing.

In [5]:
def assert_contract(client,path):
    raise NotImplementedError("Test authentication, validation, response shape, and isolation")

In [6]:
if RUN_EXERCISES:
    with TestClient(student_app) as client:
        assert_contract(client,"/student-search")
        assert_contract(client,"/search")
    print("Both endpoint contracts checked")

### 4. Run a real server from a terminal
From `labs/rag`, activate the environment and run:

```bash
export RAG_API_KEYS='{"student-alpha":"alpha","student-beta":"beta"}'
uvicorn raglab.service:create_app --factory --host 127.0.0.1 --port 58000
```

Open `http://127.0.0.1:58000/docs`, inspect the generated request schema, and call `/search`. In a second terminal:

```bash
curl -sS http://127.0.0.1:58000/search -H 'Content-Type: application/json' -H 'X-API-Key: student-alpha' -d '{"query":"E-204","k":3,"rerank":true}'
```

Stop the server with Ctrl-C. Do not expose this demonstration service publicly. To serve the student route, put it in a Python module and register it in your app factory; a function defined only inside the notebook does not modify a separate Uvicorn process.

## Checks and submission
Submit your app module and tests, plus the notebook as a client/report. Add the reference route's bounded-concurrency and sanitized 503 behavior to your own route. Test pool exhaustion/database failure by dependency injection; test a genuinely stopped database only in the isolated Compose project and restart it afterward. Explain the difference between a 401, 422, and 503. Do not log request keys or raw retrieved passages.

Rubric: route 40, contract tests 30, real HTTP demonstration 15, failure policy 15. Existing passing tests are not proof that every future SQL path preserves isolation.

## Next Steps
Lab 16 combines updates, service operation, retrieval evaluation, and optional grounded generation.

References checked 2026-10-07: [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/), [lifespan](https://fastapi.tiangolo.com/advanced/events/), [dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/), [psycopg connection pools](https://www.psycopg.org/psycopg3/docs/advanced/pool.html).